# ENN583 - Week 3 - Practical: Stereo Vision    

In this week's Prac we will explore two stereo vision algorithms available in OpenCV. 

We will again work with the -- now familiar -- Kitti dataset.



## Load the Kitti Sequence

In [ ]:
# This code cell is responsible for importing the `kitti_utils` module, which provides utilities for working with the KITTI dataset.
# It first attempts to find the repository root by looking for the presence of the `kitti_utils.py` file in the current directory and its parent directories. If it finds the file, it adds the `support` directory to the Python path so that the module can be imported. 
# If it cannot find the file, it raises a RuntimeError.
from pathlib import Path
import sys

support_dir = (Path.cwd().resolve().parents[1] / "support")
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
    
import kitti_utils as kitti

In [ ]:
# Load the KITTI sequence using the course support module.
data = kitti.load_kitti_dataset("2011_09_26_drive_0035")


import cv2
from matplotlib import pyplot as plt
import numpy as np

# get the left and right stereo images from the 10th frame
left, right = data.stereo(10)
left = np.array(left)
right = np.array(right)

# use plt to show them side by side
plt.figure(figsize=(20,10))
plt.subplot(1,2,1)
plt.imshow(left, cmap='gray')
plt.subplot(1,2,2)
plt.imshow(right, cmap='gray')
plt.show()

## Explore the camera matrix P and the matrix of intrinsic parameters K

The intrinsic matrix $K$ contains the focal lengths and principal point of the camera. The full projection matrix $P$ is a $3 \times 4$ matrix that projects a homogeneous 3D point into the image.

The KITTI projection matrix below expects the 3D point to be expressed in the **rectified camera coordinate system**. In a complete vision system, a point in a world coordinate system would first need to be transformed into this camera coordinate system.

Run the example below, then change the $X$, $Y$, and $Z$ coordinates of `point_3d`. Keep $Z$ positive so that the point remains in front of the camera. Predict how each change will move the projected pixel before running the cell again.


In [ ]:
# Get the K and P matrices for the left colour camera (camera 2).
calibration = data.camera_calibration(camera=2)
K = calibration["K"]
P = calibration["P"]

# print both matrices with 3 decimal places, suppressing small values. We use np.array2string to format the output nicely.
print("Intrinsic matrix K:\n", np.array2string(K, precision=3, floatmode='fixed', suppress_small=True))
print("\nFull projection matrix P:\n", np.array2string(P, precision=3, floatmode='fixed', suppress_small=True))

# Define a 3D point [X, Y, Z] in the rectified camera coordinate system.
# Try changing these coordinates and predict where the point will appear.
# Remember that the camera coordinate system has its origin at the camera's optical center, 
# with the Z-axis pointing forward, the X-axis pointing to the right, and the Y-axis pointing down.
# So the point [2.0, -1.0, 10.0] is 2 meters to the right, 1 meter up, and 10 meters in front of the camera.
point_3d = np.array([2.0, -1.0, 10.0])

# Add a final coordinate of 1 to make the point homogeneous: [X, Y, Z, 1].
point_homogeneous = np.append(point_3d, 1.0)

# Multiplying P by the homogeneous 3D point gives homogeneous image coordinates.
# Notice how we are using @ for matrix multiplication, which is equivalent to np.dot(P, point_homogeneous).
# It would be wrong to try using * for this, as that would perform element-wise multiplication instead of matrix multiplication.
pixel_homogeneous = P @ point_homogeneous

# Divide by the third coordinate to obtain ordinary pixel coordinates [u, v].
pixel = pixel_homogeneous[:2] / pixel_homogeneous[2]

print("\n3D point:", point_3d)
print("Homogeneous 3D point:", point_homogeneous)
print("Projected pixel [u, v]:", pixel)

# Display the projected point on the left image.
plt.figure(figsize=(12, 5))
plt.imshow(left)
plt.scatter(pixel[0], pixel[1], color="red", s=80, marker="x")
plt.title("Projection of the 3D point into camera 2")
plt.axis("off")
plt.show()


## Project the same world point into both cameras of a stereo camera system

The left and right cameras of a stereo camera system see the same 3D point at different horizontal image positions. This horizontal difference is called **disparity**.

Project the same point into cameras 2 and 3. Before running the cell, predict which camera will place the point farther to the left or right. Also predict whether the vertical pixel coordinate will change much.

Experiment with different 3D points and predict the disparity before running the cell. Change the 3D point to see how the disparity changes. Try to find a 3D point that projects to the same pixel in both cameras (i.e., zero disparity).


In [ ]:
# Get the projection matrices for the left and right colour cameras.
calibration_left = data.camera_calibration(camera=2)
calibration_right = data.camera_calibration(camera=3)
P_left = calibration_left["P"]
P_right = calibration_right["P"]

print("Left projection matrix (camera 2):\n", np.array2string(P_left, precision=3, suppress_small=True, floatmode="fixed"))
print("\nRight projection matrix (camera 3):\n", np.array2string(P_right, precision=3, suppress_small=True, floatmode="fixed"))

# Project the same homogeneous 3D point (from the previous cell) into both cameras.
pixel_left_homogeneous = P_left @ point_homogeneous
pixel_right_homogeneous = P_right @ point_homogeneous

# Convert both results from homogeneous to ordinary pixel coordinates.
pixel_left = pixel_left_homogeneous[:2] / pixel_left_homogeneous[2]

pixel_right = pixel_right_homogeneous[:2] / pixel_right_homogeneous[2]

# Rectified stereo disparity is the difference in horizontal coordinates.
projected_disparity = pixel_left - pixel_right

print("\nPixel in left image: ", pixel_left)
print("Pixel in right image:", pixel_right)
print(f"Horizontal disparity: {projected_disparity[0]} pixels")
print(f"Vertical difference:   {abs(pixel_left[1] - pixel_right[1])} pixels")

# Display the projected point in both stereo images.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(left)
axes[0].scatter(*pixel_left, color="red", s=80, marker="x")
axes[0].set_title("Camera 2")
axes[1].imshow(right)
axes[1].scatter(*pixel_right, color="red", s=80, marker="x")
axes[1].set_title("Camera 3")
for axis in axes:
    axis.axis("off")
plt.show()



## Use the Stereo Block Matcher to find Disparity
The simplest stereo matching algorithm is provided by the `StereoBM` class in OpenCV. The cell below demonstrated how to use it.

Notice that it expects the left and right image to be grey scale images, so we will have to convert the RBB images first.

You can access the documentation of the `StereoBM` here: https://docs.opencv.org/4.7.0/d9/dba/classcv_1_1StereoBM.html

In [ ]:
# load images and convert to grayscale
left, right = data.stereo(10)
left = cv2.cvtColor(np.array(left), cv2.COLOR_RGB2GRAY)
right = cv2.cvtColor(np.array(right), cv2.COLOR_RGB2GRAY)

# display both images side by side
plt.figure(figsize=(20,10))
plt.subplot(1,2,1)
plt.imshow(left, cmap='gray')
plt.title('Left Image')
plt.subplot(1,2,2)
plt.imshow(right, cmap='gray')
plt.title('Right Image')
plt.show()


# use the stereo blockmatcher to get a disparity map
stereo = cv2.StereoBM_create(blockSize=15) # this creates the blockmatcher object
disparity = stereo.compute(left, right)                     # this computes the disparity map

# display the disparity
plt.figure(figsize=(20,5))
plt.imshow(disparity,'gray')
plt.title('Disparity Map using block matching')
plt.colorbar()
plt.show()

# what is the range of disparity values?
print('Disparity range: ', disparity.min(), ' to ', disparity.max())



Notice how the disparity values range from below -16 to over 1000! We know that they should be in pixels, so what is going on here?

OpenCV returns the disparity map scaled by 16, and as a int16 type. To get the real disparity values, we will have to scale it accordingly.


In [ ]:
# this computes the disparity map
disparity = stereo.compute(left, right)                    

# scale the values in `disparity` to pixels
if disparity.dtype != np.float32:
    disparity = disparity.astype(np.float32) / 16.0

# display the disparity
plt.figure(figsize=(20,5))
plt.imshow(disparity,'gray')
plt.title('Disparity Map using block matching after scaling')
plt.colorbar()
plt.show()

# what is the range of disparity values?
print('Scaled disparity range: ', disparity.min(), ' to ', disparity.max())

That looks much better! Now we have the disparity values in pixels. 

Notice how a negative value of -1 is used to indicate pixels for which no disparity value could be found.

## Relate disparity to depth

For a rectified stereo camera, depth can be calculated from disparity using

$$Z = \frac{fB}{d},$$

where $f$ is the focal length in pixels, $B$ is the distance between the cameras in metres, and $d$ is disparity in pixels.

Notice the units: $f$ is in pixels, not in metres as you might expect, $B$ is in metres, and $d$ is in pixels. The resulting depth $Z$ will be in metres, since pixels times metres divided by pixels results in metres.

Before running the next cell, predict whether a nearby point will have a larger or smaller disparity than a distant point. Then inspect how the estimated depth changes as disparity increases.


In [ ]:
# Read the focal length from the left camera's intrinsic matrix.
# The focal length is the first element of the intrinsic matrix K, which is located at row 0, column 0.
# One common caveat: All values in K are in pixels, so the focal length is also in pixels, not in metres or mm as you might expect.
# To convert the focal length to metres, you would need to know the physical size of a pixel on the camera sensor, which is not provided in the KITTI dataset.
focal_length = calibration_left["K"][0, 0]

# The Kitti dataset provides the poses of both cameras relative to the same fixed coordinate frame on the car.
# This fixed frame (the IMU frame) is located at the center of the car, with the X-axis pointing forward, the Y-axis pointing left, and the Z-axis pointing up.
T_left_imu = calibration_left["T_cam_imu"]
T_right_imu = calibration_right["T_cam_imu"]

# These poses are 4x4 SE3 transform matrices.
print("\nRight camera pose relative to IMU:\n", np.array2string(T_right_imu, precision=4, suppress_small=True, floatmode="fixed"))
print("\nLeft camera pose relative to IMU:\n", np.array2string(T_left_imu, precision=4, suppress_small=True, floatmode="fixed"))

# Obtain the pose of the left camera relative to the right camera by multiplying the right camera's pose by the inverse of the left camera's pose.
T_right_left = T_right_imu @ np.linalg.inv(T_left_imu)

# The length of its translation component is the baseline, i.e. the distance between the two camera centres
baseline = np.linalg.norm(T_right_left[:3, 3])

print(f"\nFocal length: {focal_length:.2f} pixels")
print(f"Stereo baseline: {baseline:.3f} metres")

# Calculate depth for several example disparity values.
example_disparities = np.array([5, 10, 20, 40, 80], dtype=float)
example_depths = focal_length * baseline / example_disparities

print(f"\n{'disparity (pixels)':>18} {'depth (metres)':>16}")
for disparity_value, depth in zip(example_disparities, example_depths):
    print(f"{disparity_value:18.1f} {depth:16.2f}")

# Try adding other positive disparity values to the array above.
# What happens to depth when disparity doubles?


### Your Turn!
Let's explore some parameters of the block matcher.

In the cell below, create the stereo matcher with different parameters settings and display the resulting disparity images side by side for better comparison.

Change the following paramters:

 - `blockSize` 
   - Try different block sizes between 5 and 25. 
   - Notice that this parameter must be an odd number. Why is that?
   - What can you observe as you make the block size smaller or larger?
- Uniqueness Ratio
  - This is a post processing sep, that filters out pixels with potentially bad disparity estimates.
    - A pixel is filtered out if the best matching disparity is not sufficiently better than every other disparity in the search range, by applying a uniqueness ratio test.
  - Try changingn this value between 15 and 100, by calling `stereo.setUniquenessRatio()`
  - You can see what the current value is with `stereo.getUniquenessRatio()`
- Texture Threshold
  - This filters out areas that do not have enough texture information for reliable matching.
  - You can access it via the functions `stereo.getTextureThreshold()` and `stereo.setTextureThreshold()`

In [ ]:
# load images and convert to grayscale
left, right = data.stereo(10)
left = cv2.cvtColor(np.array(left), cv2.COLOR_BGR2GRAY)
right = cv2.cvtColor(np.array(right), cv2.COLOR_BGR2GRAY)

# YOUR TURN! Change the parameters of the blockmatcher and compare the results

# use the stereo blockmatcher to get a disparity map
# stereo = cv2.StereoBM_create(...) # try changing the blockSize
# stereo.set ...
# disparity = stereo.compute(left, right)  


# scale the disparity to proper pixel values
disparity = disparity.astype(np.float32) / 16.0

# plot the disparity map
plt.figure(figsize=(20,5))
plt.imshow(disparity,'gray')
plt.title('Disparity Map using block matching')
plt.colorbar()
plt.show()

# what is the range of disparity values?
print('Scaled disparity range: ', disparity.min(), ' to ', disparity.max())

## Try the Semi-Global Matching
OpenCV comes with a second implementation for stereo matching, based on the paper 
`Heiko Hirschmuller. Stereo processing by semiglobal matching and mutual information. Pattern Analysis and Machine Intelligence, IEEE Transactions on, 30(2):328–341, 2008.` which is available here: https://core.ac.uk/download/pdf/11134866.pdf

This is a more advanced method compared to the simple block matchers, but can be more resource hungry.

The OpenCV documentation is at https://docs.opencv.org/4.7.0/d2/d85/classcv_1_1StereoSGBM.html

In [ ]:
# Load the images. This time we do not have to convert them to grayscale, since the algorithm can work directly on RGB
left, right = data.stereo(10)
left = np.array(left)
right = np.array(right)

# create the Semi-Global Block Matching object
stereo = cv2.StereoSGBM_create(blockSize=5, numDisparities=72, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)

# compute the disparity map
disparity = stereo.compute(left, right)

# as before, we have to scale the outputs to actual pixel values
disparity = disparity.astype(np.float32) / 16.0

# display the disparity
plt.figure(figsize=(20,5))
plt.imshow(disparity,'gray')
plt.title('Disparity Map using Semi-Global Matching after scaling')
plt.colorbar()
plt.show()

# what is the range of disparity values?
print('Scaled disparity range: ', disparity.min(), ' to ', disparity.max())

### Your Turn!

The SGBM is better documented than the simple BM class. Head to https://docs.opencv.org/4.7.0/d2/d85/classcv_1_1StereoSGBM.html#adb7a50ef5f200ad9559e9b0e976cfa59 and check the parameters that can be passed into the constructor. 

Run a number of experiments comparing the effects of these parameters.

Definitely try changing `blockSize` and `numDisparities`, as well as `uniquenessRatio`, `P1` and `P2` (controlling the smoothness).

If you set `numDisparities` to a low value (it must be divisible by 16,try comparing between 16 and 128), what do you notice? Why do you think this happens?

In [ ]:
# Your Turn!



## Calculate 3D coordinates for Interest Points

We now have two different ways of establishing the 3D coordinates for interest points (such as ORB or SIFT features) in the image. 

**Option 1:**
 - We can calculate the full disparity map and use that to calculate the 3D coordiantes for specific interest points that we find in either the left or right image.


**Option 2:**
 - We can detect interest points in both the left and right image, then match between the left and right points. We then have a "sparse" disparity map, meaning we only have disparity information for the matched interest points. This can be used to calculate their 3D coordinates.

### Your Turn!
Implement both options and compare the results.

In [ ]:
# ====== OPTION 1 ======

# load images

# claculate disparity map

# find interest points in left image

# calculate 3D coordinates of interest points based on disparity map



# ====== OPTION 2 ======

# load images

# find interest points (and their descriptors) in left and right image

# match interest points between left and right image

# use difference in u-coordinates of matched interest points to calculate sparse disparity map

# calculate 3D coordinates of interest points 


# ====== Comparison ======

# Are the results of the two options the same? 

# If not, what reasons do you suspect?
